# TT-14 — ElasticNet: Dự báo tải sưởi/làm mát toà nhà

Bộ dữ liệu **Energy Efficiency (UCI)** — 768 mẫu, 8 biến thiết kế, 2 nhãn (Y1 = tải sưởi, Y2 = tải làm mát).
Đặc thù: X1, X2, X4, X5 tương quan gần như hoàn hảo (đa cộng tuyến nặng) → đúng bài toán ElasticNet.

In [1]:
import sys
sys.path.append('../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from data_loader import load_energy_data, make_design_matrix, DEFAULT_DATA_PATH

# Đổi đường dẫn này nếu bạn để dataset tự tải ở chỗ khác
DATA_PATH = "../data/ENB2012_data.csv"

df = load_energy_data(DATA_PATH)
df.head()

,X1,X2,X3,X4,X5,X6,X7,X8,Y1,Y2
0,0.98,514.5,294.0,110.25,7.0,2.0,0.0,0.0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3.0,0.0,0.0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4.0,0.0,0.0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5.0,0.0,0.0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2.0,0.0,0.0,20.84,28.28


## 1. Ma trận tương quan + VIF — xác nhận đa cộng tuyến

In [2]:
from train import compute_vif

plt.figure(figsize=(7,6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Ma trận tương quan giữa các biến thiết kế")
plt.tight_layout()
plt.show()

vif_df = compute_vif(df)
vif_df

,feature,VIF
1,X2,1.000800e+15
2,X3,1.000800e+15
3,X4,1.000800e+15
0,X1,1.055241e+02
4,X5,3.120547e+01
5,X7,1.000000e+00


VIF của X2, X3, X4 lớn bất thường (đa cộng tuyến gần như hoàn hảo — về mặt hình học,
diện tích bề mặt = hàm của diện tích tường + mái + sàn). Đây chính là lý do Lasso không phù hợp:
nó sẽ chọn ngẫu nhiên 1 biến trong nhóm và bỏ hẳn các biến còn lại.

**VIF tràn số không định lượng được mức độ** — vì X2 gần như là hệ thức hình học chính xác của X3, X4 (không chỉ tương quan cao), statsmodels không hội tụ ra số hữu hạn. Để có VIF đọc được, tính lại sau khi bỏ X2 khỏi nhóm:

In [3]:
vif_no_x2 = compute_vif(df, cols=[c for c in ["X1","X2","X3","X4","X5","X7"] if c != "X2"])
vif_no_x2

,feature,VIF
2,X4,211.938336
0,X1,105.524054
3,X5,31.205474
1,X3,27.662740
4,X7,1.000000


## 2. One-hot biến phân loại (X6, X8) + chuẩn hoá

In [4]:
X, y1, y2 = make_design_matrix(df)
feature_names = list(X.columns)
print(feature_names)
X.head()

['X1', 'X2', 'X3', 'X4', 'X5', 'X7', 'X6_3', 'X6_4', 'X6_5', 'X8_1', 'X8_2', 'X8_3', 'X8_4', 'X8_5']


,X1,X2,X3,X4,X5,X7,X6_3,X6_4,X6_5,X8_1,X8_2,X8_3,X8_4,X8_5
0,0.98,514.5,294.0,110.25,7.0,0.0,False,False,False,False,False,False,False,False
1,0.98,514.5,294.0,110.25,7.0,0.0,True,False,False,False,False,False,False,False
2,0.98,514.5,294.0,110.25,7.0,0.0,False,True,False,False,False,False,False,False
3,0.98,514.5,294.0,110.25,7.0,0.0,False,False,True,False,False,False,False,False
4,0.90,563.5,318.5,122.50,7.0,0.0,False,False,False,False,False,False,False,False


## 3-5. Baseline, Ridge/Lasso/ElasticNet, bảng so sánh (cho Y1 — tải sưởi)

In [5]:
from train import evaluate_target

cmp_y1, fitted_y1, split_y1 = evaluate_target(X, y1, "Y1_heating", feature_names)
cmp_y1

[Y1_heating] ElasticNetCV -> alpha=0.00045, l1_ratio=0.1


,target,model,n_features_kept,RMSE,R2
0,Y1_heating,Dummy,0,10.238,-0.0055
1,Y1_heating,LinearRegression,14,2.872,0.9208
2,Y1_heating,Ridge,14,2.886,0.9201
3,Y1_heating,Lasso,12,2.899,0.9194
4,Y1_heating,ElasticNetCV,14,2.876,0.9207


## 6. Hiệu ứng gom nhóm — Lasso bỏ biến nào mà ElasticNet giữ?

In [6]:
from train import grouping_effect

grp_y1 = grouping_effect(fitted_y1, feature_names)
grp_y1

,feature,Lasso_coef,ElasticNet_coef
0,X1,-4.5875,-6.2214
1,X2,-0.0451,-3.4820
2,X4,-4.9169,-3.6374
3,X5,7.9054,7.3238


Trong nhóm {X1, X2, X4, X5} tương quan chặt: Lasso đẩy hệ số của X2 gần về 0
(coi như loại bỏ), trong khi ElasticNet giữ hệ số đáng kể cho cả 4 biến — đúng
hiệu ứng gom nhóm (grouping effect) mà ElasticNet được thiết kế để tạo ra.

## 7. Heatmap RMSE theo lưới alpha × l1_ratio

In [7]:
from train import alpha_l1ratio_heatmap

alpha_l1ratio_heatmap(X, y1, "Y1_heating", "../reports/heatmap_alpha_l1ratio_Y1_heating.png")
plt.imshow(plt.imread("../reports/heatmap_alpha_l1ratio_Y1_heating.png"))
plt.axis("off")
plt.show()

## 8. Lặp lại cho Y2 (tải làm mát) — so sánh biến quan trọng giữa 2 nhãn

In [8]:
cmp_y2, fitted_y2, split_y2 = evaluate_target(X, y2, "Y2_cooling", feature_names)
grp_y2 = grouping_effect(fitted_y2, feature_names)

en_y1 = fitted_y1["ElasticNetCV"].named_steps["m"]
en_y2 = fitted_y2["ElasticNetCV"].named_steps["m"]
coef_compare = pd.DataFrame({
    "feature": feature_names,
    "coef_Y1_heating": en_y1.coef_.round(3),
    "coef_Y2_cooling": en_y2.coef_.round(3),
})
coef_compare

[Y2_cooling] ElasticNetCV -> alpha=0.00032, l1_ratio=0.1


,feature,coef_Y1_heating,coef_Y2_cooling
0,X1,-6.221,-7.180
1,X2,-3.482,-3.995
2,X3,0.873,0.221
3,X4,-3.637,-3.825
4,X5,7.324,7.214
5,X7,2.313,1.814
6,X6_3,-0.046,-0.257
7,X6_4,-0.020,-0.138
8,X6_5,-0.091,0.018
9,X8_1,1.592,0.670


Quan sát: X5 (chiều cao tổng) có hệ số dương lớn cho cả 2 nhãn — nhà càng cao,
cả tải sưởi lẫn tải làm mát càng tăng. X7 (diện tích kính) và nhóm X6 (hướng nhà)
ảnh hưởng đến tải làm mát rõ hơn tải sưởi — hợp lý về mặt vật lý (kính hấp thụ bức xạ mặt trời
làm tăng nhiệt cần làm mát vào ban ngày nhiều hơn là ảnh hưởng đến nhu cầu sưởi).

## 9. Kiểm tra ổn định hệ số bằng bootstrap (100 lần)

In [9]:
from train import bootstrap_stability

boot_y1 = bootstrap_stability(X, y1, feature_names, n_boot=100)
boot_y1

,feature,coef_mean,coef_std
1,X2,-4.4510,1.5471
3,X4,-2.8122,1.4233
2,X3,1.2387,0.6693
0,X1,-6.4472,0.6686
4,X5,7.4076,0.5639
10,X8_2,1.7370,0.2595
9,X8_1,1.7588,0.2590
11,X8_3,1.6141,0.2553
13,X8_5,1.6447,0.2440
12,X8_4,1.7059,0.2377


## 10. Đề xuất thiết kế giảm tải năng lượng

Dựa trên hệ số ElasticNet (đã chuẩn hoá) cho cả Y1 và Y2:

1. **Giảm chiều cao tổng thể (X5) khi có thể** — đây là biến có hệ số dương lớn nhất
   cho cả tải sưởi lẫn tải làm mát; giảm chiều cao trần/số tầng ở mức thiết kế cho
   phép giảm cả hai loại tải cùng lúc.
2. **Tăng độ gọn tương đối (X1 — relative compactness)** — hệ số âm mạnh ở cả hai nhãn;
   thiết kế khối nhà gọn hơn (tỉ lệ thể tích/diện tích bề mặt bao che cao hơn) giảm
   diện tích tiếp xúc với môi trường ngoài, giảm thất thoát/hấp thụ nhiệt.
3. **Kiểm soát diện tích kính (X7) và phân bố kính (X8) theo hướng nhà (X6)** —
   nhóm biến này ảnh hưởng đến tải làm mát rõ hơn tải sưởi; nên ưu tiên diện tích kính
   nhỏ hơn hoặc dùng kính cách nhiệt ở các mặt hướng nắng nhiều, thay vì phân bổ đều.